In [1]:
!pip install langchain faiss-cpu tiktoken pypdf langchain-community langchain_openapi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 79.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 329.5/329.5 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 85.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 54.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 4.0 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [3]:
!pip show langchain # to see the package info such as version

Name: langchain
Version: 1.1.0
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.12/dist-packages
Requires: langchain-core, langgraph, pydantic
Required-by: 


In [4]:
!pip install -U langchain # To update to latest version

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.1/102.1 kB 4.0 MB/s eta 0:00:00
  Attempting uninstall: langchain
    Found existing installation: langchain 1.1.0
    Uninstalling langchain-1.1.0:
      Successfully uninstalled langchain-1.1.0


In [5]:
!pip show langchain

Name: langchain
Version: 1.1.2
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.12/dist-packages
Requires: langchain-core, langgraph, pydantic
Required-by: 


In [6]:
!pip show pypdf

Name: pypdf
Version: 6.4.0
Summary: A pure-python PDF library capable of splitting, merging, cropping, and transforming PDF files
Home-page: 
Author: 
Author-email: Mathieu Fenniak <biziqe@mathieu.fenniak.net>
License: 
Location: /usr/local/lib/python3.12/dist-packages
Requires: 
Required-by: 


In [7]:
import os
from dotenv import load_dotenv
load_dotenv()
OPEN_API_KEY = os.getenv('OPEN_API_KEY')

In [8]:
OPEN_API_KEY

In [14]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_openai.llms import OpenAI

In [13]:
!pip install langchain-openai

In [15]:
pdf_path = "/content/doc202517482201.pdf"
loader = PyPDFLoader(pdf_path)
pages = loader.load()
print(f'Loaded ')

Loaded 


In [31]:
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
chunks = splitter.split_documents(pages)
print(f'Split into {len(chunks)} chunks')
print ("\n Samle Chunnk \n", chunks[16].page_content[:500])


Split into 47 chunks

 Samle Chunnk 
 Cement and Consumption of Steel, (vii) Net Tonne Kilometres and Passenger Kilometres  for 
Railways, (viii) Passenger and Cargo traffic handled by Civil Aviation, (ix) Cargo traffic handled at 
Major and Minor Sea Ports, (x) Sales of Commercial Vehicles, (xi) Bank Deposits and Credits, (xii) 
Premium related information of Life and Non -Life Insurance companies, (xiii) Data on outward 
Supplies of Goods and Services available from GSTN  upto November 2024 , (xiii) Accounts of


In [19]:
embeddings = OpenAIEmbeddings()
vectorstore = FAISS.from_documents(chunks, embeddings)
print(f'Vectorstore: loaded')

Vectorstore: loaded


In [25]:
query = "What is the GDP growth rate for India for 2024-25?"

from openai import OpenAI
client = OpenAI()

In [29]:
# With RAG

results = vectorstore.similarity_search(query, k=3)
context = "\n\n".join([doc.page_content for doc in results])
prompt = f'''
Use the following document context to answer the question.
Context: {context}
---
Question: {query}
Answer:
'''

rag_response = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[
        {"role": "system", "content": "You are a helpful assistant"},
        {"role": "user", "content": prompt}
    ]
   )
print("GDP. with RAG")
print(rag_response.choices[0].message.content)

GDP. with RAG
The GDP growth rate for India for 2024-25 is estimated to be 6.4% based on the information provided in the document context.


In [32]:
print(context)

lakh crore. The growth rate in Real GDP during 2024-25 is estimated at 6.4% as compared to 8.2% 
in 2023-24. Nominal GDP or GDP at Current Prices is estimated to attain a level of ₹ 324.11 lakh 
crore in the year 2024-25, against ₹295.36 lakh crore in 2023-24, showing a growth rate of 9.7%.  
 Real GVA is estimated at ₹168.91 lakh crore in the year 2024-25, against the PE for the year 
2023-24 of ₹158.74 lakh crore, registering a growth rate of 6.4% as compared to 7.2% growth rate

This Press Release is embargoed against publication, telecast or circulation on internet till 4:00 pm on 7th January, 2025. 
 
2 
 
I. Annual GDP Estimates and Growth Rates 
 
 Real GDP or GDP at Constant Prices is estimated to attain a level of ₹ 184.88 lakh crore in 
the financial year 2024-25, against the Provisional Estimate of GDP for the year 2023-24 of ₹173.82 
lakh crore. The growth rate in Real GDP during 2024-25 is estimated at 6.4% as compared to 8.2%

2023-24 of ₹158.74 lakh crore, registering a 

In [30]:
# Without using RAG
from openai import OpenAI
client = OpenAI()
query = "What is the GDP growth rate for India for 2024-25?"
direct_response = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[
        {"role": "system", "content": "You are a helpful assistant"},
        {"role": "user", "content": query}
    ]
)

print("GDP. without RAG")
print(direct_response.choices[0].message.content)


GDP. without RAG
I'm sorry, but I don't have information on the specific GDP growth rate for India for the year 2024-25. Economic projections and forecasts are typically made by government agencies, financial institutions, and research organizations. You may want to refer to reports from organizations such as the International Monetary Fund (IMF), World Bank, or the Reserve Bank of India for more accurate and up-to-date information on GDP growth projections for India in 2024-25.


In [33]:
import tiktoken

def count_tokens(text: str, model:str='gpt-3.5-turbo'):
  encoding = tiktoken.encoding_for_model(model)
  num_tokens = len(encoding.encode(text))
  return num_tokens
#

In [34]:
prompt=f'''
Use the following document context to answer the question.
Context: {context}
Question: {query}
Answer:
'''
rag_tokens = count_tokens(prompt)
print(f"Number of tokens in prompt: {rag_tokens}")
rag_cost = (rag_tokens/1000)*0.002
print(f"Cost of prompt: {rag_cost}")

Number of tokens in prompt: 433
Cost of prompt: 0.000866
